In [ ]:
%pwd

In [ ]:
import os 
os.chdir("../")

In [ ]:
%pwd

In [ ]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [ ]:
# Extract text from PDF files
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()
    return documents

In [ ]:
extracted_data = load_pdf_files("data")

In [ ]:
extracted_data

In [ ]:
len(extracted_data)

In [ ]:
from typing import List
from langchain_core.documents import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:

    """
    Given a list of Document objects, return a new list of Document objects
    containing only 'source' in metadata and the original page_content.
    """

    minimal_docs: List[Document] = []

    for doc in docs:
        src = doc.metadata.get("source")

        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )

    return minimal_docs

print("Function loaded successfully!")

In [ ]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [ ]:
minimal_docs

In [ ]:
# Split the documents into smaller chunks
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [ ]:
texts_chunk = text_split(minimal_docs)
print(f"Number of chunks: {len(texts_chunk)}")

In [ ]:
texts_chunk

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    """
    Download and return the HuggingFace embeddings model.
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
    )
    return embeddings

embedding = download_embeddings()

In [ ]:
embedding

In [ ]:
vector = embedding.embed_query("Hello world")
vector

In [ ]:
print("Vector length:", len(vector))

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

In [ ]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")



os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [ ]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [ ]:
pc

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

print("Pinecone key loaded:", PINECONE_API_KEY is not None)
print("OpenAI key loaded:", OPENAI_API_KEY is not None)

In [ ]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

index = pc.Index(index_name)

print("Pinecone index is ready!")

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    
    documents = loader.load()
    return documents

In [ ]:
extracted_data = load_pdf_files("data")

print("Documents loaded:", len(extracted_data))

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=20
)

texts_chunk = text_splitter.split_documents(extracted_data)

print("Text chunks created:", len(texts_chunk))

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=20
)

texts_chunk = text_splitter.split_documents(extracted_data)

print("Text chunks created:", len(texts_chunk))

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

In [ ]:
index_name = "medical-chatbot"

print("index_name:", index_name)

In [ ]:
print("extracted_data:", len(extracted_data))
print("texts_chunk:", len(texts_chunk))
print("embedding:", embedding)
print("index_name:", index_name)

In [ ]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name
)

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

print("Pinecone key loaded:", PINECONE_API_KEY is not None)

In [ ]:
from pinecone import Pinecone
import os

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

index = pc.Index(index_name)

stats = index.describe_index_stats()

print(stats)

In [ ]:
print("Total chunks:", len(texts_chunk))
print("Vectors already in Pinecone:", stats["total_vector_count"])
print("Approximate remaining:", len(texts_chunk) - stats["total_vector_count"])


In [ ]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name,
    batch_size=50
)

print("All documents successfully stored in Pinecone!")

In [ ]:
#Load Existing index

from langchain_pinecone import PineconeVectorStore
#Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

# Add more data to the existing Pinecone Index

In [ ]:
dswith = Document(
    page_content = "hello my name is vrund mirani.",
    metadata={"source": "Linkedin"}
)

In [ ]:
docsearch.add_documents(documents=[dswith])

In [ ]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [ ]:
retrieved_docs = retriever.invoke("what is Acne?")
retrieved_docs

In [ ]:
from dotenv import load_dotenv
import os
from pathlib import Path

env_path = Path("../.env")

print("Looking for:", env_path.resolve())
print("Exists:", env_path.exists())

load_dotenv(env_path)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

print("OpenAI key loaded:", OPENAI_API_KEY is not None)

In [ ]:
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print("OPENAI_API_KEY set:", "OPENAI_API_KEY" in os.environ)

In [ ]:
from langchain_ollama import ChatOllama

chatModel = ChatOllama(
    model="llama3.2:3b",
    temperature=0.2
)

print("Local AI model connected successfully!")

In [ ]:
from pathlib import Path

print("Current folder:", Path.cwd())
print("Env file exists:", Path(".env").exists())
print("Parent env exists:", Path("../.env").exists())


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("../.env", override=True)

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

print("Pinecone key loaded:", bool(PINECONE_API_KEY))

In [ ]:
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)

index = pc.Index("medical-chatbot")

print(index.describe_index_stats())

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model ready!")

In [ ]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore(
    index=index,
    embedding=embedding
)

retriever = docsearch.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever ready!")

In [ ]:
docs = retriever.invoke(
    "What are acromegaly and gigantism?"
)

print("Documents retrieved:", len(docs))

for i, doc in enumerate(docs, start=1):
    print(f"\nDocument {i}:")
    print(doc.page_content[:500])

In [ ]:
from langchain_ollama import ChatOllama

chatModel = ChatOllama(
    model="llama3.2:3b",
    temperature=0.2
)

print("Local AI model connected!")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Answer the user's question using the provided medical context.

Context:
{context}

Question:
{input}

Instructions:
- Give a clear, easy-to-understand answer.
- If the context does not contain enough information, say so.
- Do not invent medical facts.
- Explain that the chatbot is for educational purposes and is not a substitute for professional medical advice.
""")

print("Medical prompt ready!")

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

question_answer_chain = create_stuff_documents_chain(
    chatModel,
    prompt
)

rag_chain = create_retrieval_chain(
    retriever,
    question_answer_chain
)

print("Medical RAG chain ready!")

In [ ]:
response = rag_chain.invoke({
    "input": "What are acromegaly and gigantism?"
})

print(response["answer"])

In [ ]:
from pathlib import Path

project_root = Path.cwd().parent

prompt_code = '''from langchain_core.prompts import ChatPromptTemplate

MEDICAL_SYSTEM_PROMPT = """You are a medical information assistant.

Answer the user's question using the supplied medical reference context.

Rules:
1. Explain medical information clearly and accurately.
2. Ground factual answers in the supplied context.
3. If the context is insufficient, explicitly say so.
4. Never invent citations, medical facts, or source details.
5. Do not claim to diagnose conditions or prescribe treatment.
6. Encourage users to consult qualified healthcare professionals
   for personal medical concerns.
7. For potentially life-threatening symptoms, advise seeking
   emergency medical care immediately.
8. Make clear that this assistant is educational and does not
   replace professional medical advice.

Reference context:
{context}

User question:
{input}
"""

medical_prompt = ChatPromptTemplate.from_template(
    MEDICAL_SYSTEM_PROMPT
)
'''

prompt_path = project_root / "src" / "prompt.py"
prompt_path.write_text(prompt_code, encoding="utf-8")

print("Created:", prompt_path)
print("File size:", prompt_path.stat().st_size, "bytes")
print("Prompt file restored successfully!")

In [ ]:

from pathlib import Path

project_root = Path.cwd().parent
requirements_path = project_root / "requirements.txt"

print(requirements_path.read_text(encoding="utf-8"))

print("\nMalformed '- e' entry present:",
      "- e" in requirements_path.read_text(encoding="utf-8").splitlines())

print("Hugging Face integration listed:",
      "langchain-huggingface==0.3.1"
      in requirements_path.read_text(encoding="utf-8").splitlines())

print("Ollama integration listed:",
      "langchain-ollama==0.3.10"
      in requirements_path.read_text(encoding="utf-8").splitlines())


In [ ]:

from pathlib import Path

project_root = Path.cwd().parent
requirements_path = project_root / "requirements.txt"

new_requirements = """langchain==0.3.26
flask==3.1.1
sentence-transformers==4.1.0
pypdf==5.6.1
python-dotenv==1.2.4
langchain-pinecone==0.2.8
langchain-community==0.3.26
langchain-huggingface==0.3.1
langchain-ollama==0.3.10
"""

requirements_path.write_text(
    new_requirements,
    encoding="utf-8"
)

saved_content = requirements_path.read_text(encoding="utf-8")

assert saved_content == new_requirements
assert "- e" not in saved_content.splitlines()

print("SUCCESS: requirements.txt updated and verified.")
print("\nSaved contents:")
print(saved_content)


In [ ]:

from pathlib import Path

project_root = Path.cwd().parent
init_path = project_root / "src" / "__init__.py"

print("File:", init_path)
print("Exists:", init_path.exists())

if init_path.exists():
    content = init_path.read_text(encoding="utf-8")
    print("Size:", len(content), "characters")
    print("Contents:")
    print(repr(content))
else:
    print("The file does not exist.")


In [ ]:

from pathlib import Path

project_root = Path.cwd().parent

files_to_inspect = [
    "src/helper.py",
    "src/prompt.py",
]

for relative_path in files_to_inspect:
    file_path = project_root / relative_path

    print("\n" + "=" * 70)
    print("FILE:", relative_path)

    if not file_path.exists():
        print("NOT FOUND")
        continue

    content = file_path.read_text(encoding="utf-8")
    print("SIZE:", len(content), "characters")
    print("-" * 70)
    print(content)


In [ ]:

from pathlib import Path

project_root = Path.cwd().parent

for relative_path in ["src/helper.py", "src/prompt.py"]:
    file_path = project_root / relative_path

    print("\n" + "=" * 70)
    print(f"FILE: {relative_path}")

    if not file_path.exists():
        print("ERROR: File not found")
        continue

    lines = file_path.read_text(encoding="utf-8").splitlines()

    print(f"Total lines: {len(lines)}")

    for start in range(0, len(lines), 35):
        end = min(start + 35, len(lines))
        print(f"\n--- Lines {start + 1}-{end} ---")

        for line_number in range(start, end):
            print(f"{line_number + 1:03}: {lines[line_number]}")


In [ ]:

from pathlib import Path

project_root = Path.cwd().parent

source_files = [
    "src/helper.py",
    "src/prompt.py",
]

report_parts = []

for relative_path in source_files:
    file_path = project_root / relative_path

    report_parts.append(f"\n{'=' * 70}")
    report_parts.append(f"FILE: {relative_path}")
    report_parts.append("=" * 70)

    if file_path.exists():
        report_parts.append(
            file_path.read_text(encoding="utf-8")
        )
    else:
        report_parts.append("FILE NOT FOUND")

report_path = project_root / "source_inspection.txt"
report_path.write_text(
    "\n".join(report_parts),
    encoding="utf-8"
)

print("Report created:", report_path)
print("Report size:", report_path.stat().st_size, "bytes")
print("\nFirst 20 lines:")
print("\n".join(report_path.read_text(encoding="utf-8").splitlines()[:20]))


In [ ]:

from pathlib import Path

project_root = Path.cwd().parent

# 1. Display environment variable NAMES only.
env_path = project_root / ".env"

print("========== ENVIRONMENT VARIABLE NAMES ==========")

if env_path.exists():
    names = []

    for line in env_path.read_text(
        encoding="utf-8"
    ).splitlines():
        line = line.strip()

        if line and not line.startswith("#") and "=" in line:
            name = line.split("=", 1)[0].strip()
            if name:
                names.append(name)

    for name in sorted(set(names)):
        print(name)

    if not names:
        print("No variable assignments found.")
else:
    print(".env file not found.")

# 2. Check relevant .gitignore rules without dumping the file.
gitignore_path = project_root / ".gitignore"

print("\n========== SECRET / ENV IGNORE RULES ==========")

if gitignore_path.exists():
    lines = gitignore_path.read_text(
        encoding="utf-8"
    ).splitlines()

    relevant = [
        line for line in lines
        if ".env" in line.lower()
        or "secret" in line.lower()
        or "credential" in line.lower()
    ]

    print("\n".join(relevant) if relevant else
          "No relevant rules found.")
else:
    print(".gitignore not found.")

# 3. Check whether Git currently tracks .env.
print("\n========== GIT TRACKING CHECK ==========")

import subprocess

result = subprocess.run(
    ["git", "ls-files", "--error-unmatch", ".env"],
    cwd=project_root,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("WARNING: .env is tracked by Git.")
else:
    print(".env is not tracked by Git (or Git metadata is unavailable).")


In [ ]:

from pathlib import Path

project_root = Path.cwd().parent
app_path = project_root / "app.py"

content = app_path.read_text(encoding="utf-8")

print("File:", app_path)
print("Size:", len(content), "characters")
print("Has Flask app:", "app = Flask(__name__)" in content)
print("Uses local Ollama:", 'model="llama3.2:3b"' in content)
print("Uses existing Pinecone index:",
      'Index("medical-chatbot")' in content)
print("Contains chat route:", '@app.post("/chat")' in content)
print("Contains health route:", '@app.get("/health")' in content)


In [ ]:
import ast
import sys
from pathlib import Path

project_root = Path.cwd().parent
app_path = project_root / "app.py"

# Check Python syntax without executing the application
source_code = app_path.read_text(encoding="utf-8")
ast.parse(source_code, filename=str(app_path))

print("1. Python syntax: PASS")

# Check imports without initializing the RAG pipeline
import app

print("2. App import: PASS")
print("3. Flask application:", app.app.name)
print("4. Chat route:", app.app.view_functions.get("chat") is not None)
print("5. Health route:", app.app.view_functions.get("health") is not None)
print("6. RAG initialized:", app.rag_chain is not None)

In [ ]:
import subprocess
import shutil

ollama_path = (
    shutil.which("ollama") or "ollama"
)

result = subprocess.run(
    [ollama_path, "list"],
    capture_output=True,
    text=True,
    timeout=30
)

print("Return code:", result.returncode)
print("Models:")
print(result.stdout)

if result.stderr:
    print("Errors:")
    print(result.stderr)

In [ ]:
import app

try:
    app.initialize_rag()
    print("RAG initialization: PASS")
    print("RAG chain created:", app.rag_chain is not None)
except Exception as exc:
    print("RAG initialization: FAILED")
    print("Error type:", type(exc).__name__)
    print("Error:", str(exc))

In [ ]:
question = "What are acromegaly and gigantism?"

try:
    result = app.rag_chain.invoke({"input": question})

    print("QUESTION:")
    print(question)

    print("\nANSWER:")
    print(result.get("answer", "No answer returned."))

    print("\nRETRIEVED DOCUMENTS:")
    documents = result.get("context", [])
    print("Count:", len(documents))

    for i, document in enumerate(documents, start=1):
        print(f"\n--- Source {i} ---")
        print("File:", document.metadata.get("source"))
        print("Page:", document.metadata.get("page_label"))
        print("Excerpt:", document.page_content[:400])

except Exception as exc:
    print("RAG test failed.")
    print("Error type:", type(exc).__name__)
    print("Error:", str(exc))

In [ ]:
import app

client = app.app.test_client()

# Test 1: Health endpoint
health_response = client.get("/health")

print("HEALTH TEST")
print("Status:", health_response.status_code)
print("Response:", health_response.get_json())

# Test 2: Valid medical question
chat_response = client.post(
    "/chat",
    json={"question": "What are acromegaly and gigantism?"}
)

print("\nCHAT TEST")
print("Status:", chat_response.status_code)

chat_data = chat_response.get_json()

print("Answer received:", bool(chat_data.get("answer")))
print("Sources returned:", len(chat_data.get("sources", [])))
print("Disclaimer present:", bool(chat_data.get("disclaimer")))

# Test 3: Empty question should be rejected
invalid_response = client.post(
    "/chat",
    json={"question": "   "}
)

print("\nINVALID INPUT TEST")
print("Status:", invalid_response.status_code)
print("Response:", invalid_response.get_json())

In [ ]:
import subprocess

commands = [
    [
        "powershell",
        "-NoProfile",
        "-Command",
        "Get-NetTCPConnection -LocalPort 5000 -State Listen "
        "-ErrorAction SilentlyContinue | "
        "Select-Object LocalAddress,LocalPort,OwningProcess"
    ],
    [
        "powershell",
        "-NoProfile",
        "-Command",
        "Get-CimInstance Win32_Process "
        "-Filter \"name = 'python.exe'\" | "
        "Select-Object ProcessId,CommandLine | Format-List"
    ]
]

for i, command in enumerate(commands, start=1):
    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        timeout=15
    )
    print(f"\n--- Diagnostic {i} ---")
    print(result.stdout or "(no output)")
    if result.stderr:
        print(result.stderr)